# Lab 3b — LLM Mental Health Counselling Training App

**Course**: CSE 498 <br>
**Assignment**: Building an LLM Mental Health Training App (counsellors practise against simulated patients) <br>
**Base codebase**: [efuayankey/aimes](https://github.com/efuayankey/aimes) — "AIMES" / "CALLM-CARE", authored by **Efua Yankey** <br>
**My fork**: [shinnew9/aimes](https://github.com/shinnew9/aimes) <br>
**My work for this lab**: branch [`feat/returning-patients-session-memory`](https://github.com/shinnew9/aimes/tree/feat/returning-patients-session-memory) <br>
**Reviewable diff**: [compare view](https://github.com/shinnew9/aimes/compare/fix/spanish-english-translation-toggle...feat/returning-patients-session-memory)

---

## 0. Summary
This lab was **not** built from scratch. It was implemented as an extension of the AIMES counselling-training app already in use in the lab, rather than as a new standalone project.

The app has **three separate conversation modes**, and they do not behave alike. An earlier draft of this report did not distinguish them, and so stated more broadly than the evidence supports that patients do not persist. §1.1 sets the modes out and corrects that.

| Mode | Sessions stored | Counsellor can resume with the same person |
|---|---|---|
| **Real patients** (student ↔ counsellor) | Yes | **Yes** |
| **Simulated patients** (the training mode) | Yes, transcripts stored in full | **No** |
| Legacy message queue | Yes | n/a — ticket-and-reply, not a live thread |

**All three modes are Efua's**, including the simulated-patient trainer itself — the persona generation, the roleplay prompt, the session flow and the feedback rubric were all already there. What was missing was continuity *inside* the simulated trainer: a counsellor could not return to a simulated patient they had already met. That is the layer I added.

Everything below concerns the simulated trainer only. Against the brief, two of its seven requirements were unmet, and I built both.

| Requirement | Already in the codebase | My work |
|---|---|---|
| Login / register | Yes (Firebase Auth) | — |
| Real vs. simulated patient split | Yes | — |
| **New simulated-patient profile selection** | **Partial** | **Completed** |
| **Choosing a simulated patient spoken to before** | **No** | **Built** |
| Profile summary, patient speaks first | Yes | Extended |
| Multi-turn session with an explicit end | Yes | — |
| **Past simulated sessions carried forward** | **Stored, never read back** | **Built** |
| Feedback on five dimensions + overall + suggestions | Yes | — |

**Size of the change**: 10 files, **1,403 lines added, 87 removed**. Nothing in the original codebase was rewritten or deleted.

**Key points**

1. In the simulated trainer, every session generated a new persona. Finished sessions **were** written to Firestore with the complete transcript, but
   nothing ever read one back: `getTrainingSession(sessionId)` exists in the service and has **no callers anywhere in the codebase**. So the data was
   kept and the "existing patients" path still had nothing behind it.
2. A finished session is reduced to a **structured record of nine fields**, not a prose recap. The reasoning for that choice is in §3 and is the part 
   of this lab that required the most thought.
3. The record is used twice: shown to the counsellor **before** the next session, and included in the patient prompt **during** it.
4. The profile screen now takes a **list** of concerns and a proper **age group**. Previously it accepted one concern, and its age
   selector offered only 18-26 in three-year bands.
5. The change adds **zero new type errors** against a stashed baseline — it removes one — and the production build passes.
6. Two pre-existing issues surfaced along the way and are documented in §5: the Firestore rules gate on a custom claim that nothing sets, and the
   OpenAI key is shipped to the browser in the client bundle.

## 1. Starting Point — Auditing the Existing Codebase

The brief specifies a feature set. Before writing anything I went through the codebase to establish
what was already there, since duplicating working features would have been wasted effort.

The app is **Next.js 15 + TypeScript**, with **Firebase Authentication** and **Firestore**, and
**OpenAI `gpt-4o`** behind `src/app/api/*` routes. Roughly 21,500 lines of TypeScript.

### 1.1 Correction — the app has three conversation modes

My first pass through the codebase reported, without qualification, that patients do not persist. That
is too broad. `CounselorDashboard.tsx:161` switches between three modes, and only one of them lacks a
resume path:

| Mode | Stored where | Resume path | Verdict |
|---|---|---|---|
| **Real patients** | `conversations`, `conversation_messages`, `conversation_participants` | `CounselorDashboard.tsx:131-142` — the **"My Active Conversations"** tab renders an **"Open Chat"** button, which mounts `ContinuousChat`; `ContinuousChatInterface.tsx:76` then loads the entire prior transcript | **Persists and resumes** |
| **Simulated patients** | `trainingSessions` — `saveTrainingSession` writes the whole `messages` array verbatim (`trainingSessionService.ts:34-37`) | none | **Persists, cannot resume** |
| Legacy message queue | `messages`, `responses` | n/a — ticket-and-reply | not a live thread |

So a counsellor **can** return to a real student they have spoken to before; that was already built and
I should have said so. What was missing is the equivalent for *simulated* patients — not the trainer,
which Efua built, but the ability to meet the same simulated patient twice. That is what the rest of
this report is about.

One caveat found while checking: `getCounselorConversations` filters on `status == 'active'`
(`conversationService.ts:253-257`), so once a thread is marked complete it disappears from the
counsellor's list with no way back to it. The data survives; the affordance does not.

| # | Requirement from the brief | Status found | Evidence |
|---|---|---|---|
| 1 | Home page with login / register | Implemented | `src/components/auth/` — `LoginForm`, `SignupForm`, `ForgotPasswordForm`, `AuthContext` |
| 2 | Second page: "real patients" / "simulated patients" | Implemented | `PatientModeSelector`, `PatientMode = 'real' \| 'simulated' \| 'cbt'` |
| 3a | New patient: gender, age group, ethnicity, **list of** concerns | **Partial** | Gender and ethnicity are fully supported. Concern is **single-select** (`concern?: MentalHealthConcern`, not an array). An age selector exists, but it offers only `18-20 / 21-23 / 24-26 / random` — college age in three-year bands, not an age *group* in the sense the brief means |
| 3b | **Choose from simulated patients spoken to before** | **Not implemented** | `generateRandomPatient()` is called on every session start (`SimulatedSessionInterface.tsx:158`), and the persona gets a throwaway identity — `id: \`sim_${Date.now()}_${Math.random()...}\`` (`patientSimulationService.ts:52`), with name, traits and trust level all re-rolled. Picking the same culture, gender and concern yields a different person. No `simulatedPatients` collection exists |
| 4 | Session page shows the profile; patient speaks first | Implemented | `showPatientInfo`, `generateOpeningMessage()` |
| 5 | Multi-turn until the counsellor ends the session | Implemented | Also has automatic end detection (`SessionEndDetectionService`), which exceeds the brief |
| 6 | **Store past sessions; show a summary next time** | **Stored, never read back** | Storage works and keeps the full transcript. The read side does not exist — see below |
| 7 | Feedback: cultural sensitivity, therapeutic progress, professional boundaries, empathy, conversational flow, overall, suggestions | Implemented | `ConversationFeedback.overallPerformance` has exactly those five plus `overallScore` |

### Why requirement 6 counted as unmet even though sessions are stored

Sessions are saved, and saved generously: `saveTrainingSession` (`trainingSessionService.ts:27-45`)
writes `{...session}`, which carries both the whole `simulatedPatient` object and the complete
`messages` array, with no summarisation.

The gap is entirely on the read side. **`getTrainingSession(sessionId)` — the one function that could
rehydrate a stored session — has zero callers anywhere in `src/`.** Every call site of the training
service is a write, a display, a re-analysis or an export:

| Call site | What it does |
|---|---|
| `SimulatedSessionInterface.tsx:158` | `generateRandomPatient()` — fresh patient |
| `SimulatedSessionInterface.tsx:313` | `saveTrainingSession()` — write only |
| `TrainingHistoryView.tsx:44,45,62,98,113` | read for display / re-analyse / export |
| `CounselorFeedbackDashboard.tsx:375` | aggregate statistics only |

`TrainingHistoryView` is the one screen that surfaces past sessions, and its only per-session actions
are **View Details**, **Re-analyze** and **Export** — none of which hands anything back to a live session.

A second thing that looked like the feature is `generateSessionSummary()`. It is not:

```
sessionTopics:         keyword matching — if the text contains 'anxious'/'worried' → tag 'anxiety'
emotionalProgression:  simplified heuristic
duration, messageCount, averageResponseLength
```

It produces **keyword tags and message statistics**, and a search of the whole codebase for
`previousSession`, `sessionSummary`, `continuity`, `lastSession` and similar found **no path by
which its output reaches the next session's prompt**. It feeds the end-of-session detection UI only.

### Requirements 3b and 6 are one problem

A patient cannot be "existing" unless something about them persists, and a summary of a previous
session is meaningless without a patient to attach it to. They were treated as a single piece of work.

### 1.2 A related finding — the within-session context cap

While tracing how much history reaches the model, the prompt builder turned out to truncate hard
(`patientSimulationService.ts:163`):

```js
const historyText = conversationHistory
  .slice(-4) // Last 4 messages for context
```

So a simulated patient sees **only the last four messages**, and forgets anything said more than two
exchanges earlier — *within a single session*, before cross-session memory is even in question. Stored
transcripts therefore cost no prompt tokens at all, because they never reach a prompt; the real
inefficiency is in the opposite direction.

This cap is **still in place**. The continuity record added in §2 is appended to the system prompt and is
not subject to it, so it survives a long session — but the turn-by-turn history does not. Raising the cap,
and measuring what it costs, is the obvious next piece of work.

## 2. What Was Implemented

### 2.1 Patients persist between sessions

A generated persona is stored per counsellor in a new Firestore collection, `simulatedPatients`,
and **reused verbatim** on a return visit.

Reusing rather than regenerating matters more than it first appears: `generateRandomPatient()`
derives the backstory, personality traits and trust level from the profile options, so regenerating
from the same options would produce a *different person with the same demographics* every visit —
the opposite of continuity.

The patient is written **when the session starts**, not when it ends, so a session abandoned halfway
still leaves someone to come back to.

### 2.2 A finished session is distilled into a carry-forward record

At the end of a session the transcript is sent to `gpt-4o` through a new route,
`src/app/api/session-memory/route.ts`, and reduced to the nine fields described in §3.

Three properties of this step were treated as requirements rather than nice-to-haves:

- **No invention.** The prompt forbids recording any detail not present in the transcript.
- **Risk is never softened.** Risk indicators may not be omitted or rephrased away.
- **Failure must not lose data.** If extraction fails, a minimal locally-derived record is stored
  and the session is still saved. Losing the session because the summary failed would be worse than
  storing a thin summary.

### 2.3 The profile screen now matches the brief

The brief asks the counsellor to choose *"gender, age-group, ethnicity and a **list** of mental health concerns"*. Gender and ethnicity were already fine. The other two were not.

**Concerns: single-select to multi-select.** `PatientGenerationOptions.concern` held one value, so a patient could only ever present with one thing. It is now a list, and the order matters: the first concern selected is what the client came in for, and the rest are given to the model with an explicit instruction not to volunteer them —

> *You are also dealing with X, Y, but you do not volunteer these unless the counselor creates room for them — they come up sideways, or when something they say touches one of them.*

That instruction is the point of the change. A patient who recites three problems in the opening turn is easier to counsel than a real one, not harder.

To avoid a risky refactor the singular `mentalHealthConcern` field was **kept as the primary concern**, because the backstory, session goals, trust level and opening lines are all keyed on a single value. The full list lives alongside it in `mentalHealthConcerns`, read through a helper (`concernsOf`) that falls back to the singular field, so patients saved before this change still render correctly.

**Age range to age group.** The old selector offered `18-20 / 21-23 / 24-26 / random`. It now offers five groups spanning adolescent to older adult, with the exact age drawn from the chosen span:

| Group | Span |
|---|---|
| Adolescent | 13-17 |
| Young adult | 18-25 |
| Adult | 26-39 |
| Middle-aged | 40-59 |
| Older adult | 60+ |

![Profile selection](screenshots/crop-profile-selection.png)

Four concerns are selected and the badges number them in the order chosen; the profile summary
underneath resolves that to `Primary: Anxiety`.

This exposed a bug the 18-26 ceiling had been hiding. The persona prompt opened with `a ${patient.age}-year-old ${patient.gender} college student` — *college student* hardcoded. Widening the age range would have produced sixty-five-year-old college students. The prompt now derives a life stage from the age group instead.

### 2.4 The summary is shown as soon as the session ends

In the first version the extracted record went straight to the patient's file, and the counsellor saw
nothing at all until their next session with that patient. Running the app end to end made the problem
obvious: the feature worked, but from the counsellor's seat it looked as though nothing had happened.

A **Session Summary panel** now renders directly beneath the transcript it was made from, showing the
headline, risk indicators, emotional state at close, and the carried-forward fields, with the rapport
level as a badge. Risk is stated either way — an empty risk list is itself information, because it
tells the counsellor the check was made rather than skipped.

![Session summary panel](screenshots/crop-summary-panel.png)

The full-page capture this is cropped from, showing the transcript it was extracted from, is in §7.2.

### 2.5 The record is used in two places

| Where | What it does |
|---|---|
| **Before** the next session | The roster screen shows the previous summary, unresolved threads and any risk indicators, so the counsellor can prepare the way they would for a real returning client instead of rediscovering everything mid-conversation |
| **During** the next session | The record is appended to the patient's system prompt, rebuilt on every turn so it survives re-renders and is not dropped once the conversation gets long |

![Returning patient card](screenshots/crop-returning-card.png)

Jordan's card, with the session history expanded. Two sessions, rapport recorded as **High**.

The rapport level is also what sets the opening turn of the next session, chosen by the application
before the model is called:

![Second session opening](screenshots/crop-second-opening.png)

With low rapport recorded, the same patient would have opened with *"Hi."* and little else.

The continuity block is phrased as the patient's own recollection, and explicitly instructs against
reciting it:

> *Refer to these things the way a person does — in passing, half-remembered — never as a list or a
> recap. Do NOT summarise previous sessions. Do not say "last time we discussed...".*

Without that instruction the model opens a return visit by reading its own case file aloud.

## 3. Design Decision — What Counts as an "Important Fact"

The brief asks the student to *"think through what are important facts"* to store. This section is
that reasoning.

### Why not a prose summary

A paragraph summary is the obvious choice and was rejected. The problem is what happens downstream:
a paragraph can only be used one way — pasted whole into the next prompt. Once pasted, the model
treats it as material to narrate, and the patient opens the next session by summarising their own
previous session, which no real client does.

Discrete fields can be used **selectively and differently**:

- `rapportLevel` steers the greeting — it never appears in the prompt as text.
- `unresolvedThreads` steers the content.
- `riskFlags` are rendered in the UI by the application, **whether or not the model mentions them**.

A paragraph cannot be wired to three different consumers. The schema is the feature.

### The nine fields, and why each earns its place

The test applied to each candidate field was: *does this change how the next session should open?*
Anything that did not was left out.

| Field | Why it is kept |
|---|---|
| `headline` | One line, readable in the three seconds before a session starts |
| `presentingConcerns` | What was **actually** discussed, which routinely differs from the profile on file |
| `keyDisclosures` | Said for the first time. The patient will expect it to be remembered; asking again reads as not listening |
| `emotionalState` | Where the session **landed**, not where it started |
| `riskFlags` | Self-harm, hopelessness, safeguarding. The one field that must never be lost between sessions |
| `counselorCommitments` | "We agreed you'd try X." Unaddressed next time, the counsellor looks unreliable |
| `unresolvedThreads` | Gives the next session a place to begin |
| `culturalNotes` | Culture, faith, family and language factors that shaped the session |
| `rapportLevel` | Sets the patient's opening posture on return |

### Fields deliberately excluded

| Excluded | Reason |
|---|---|
| Message counts, duration, average response length | Already computed elsewhere, and they change nothing about how the next session opens |
| A diagnosis or clinical impression | The app is a training simulator, not a clinical record. Generating a diagnosis would invite it being read as one |
| Verbatim transcript quotes | The full transcript is stored anyway; duplicating it into the summary makes the prompt long without adding usable structure |

### How `rapportLevel` is consumed

This is the clearest example of a field doing work that prose could not:

| Recorded rapport | Opening behaviour on return |
|---|---|
| `low` | Guarded. "Hi." / "I'm here. I don't really know what else to say." |
| `medium` | Polite but testing. "Hi again. It's been a week, I guess." |
| `high` | Warmer and more direct. "Hey. I've actually been looking forward to this one." |

The opening line is chosen by the application from the recorded value, so the first turn of a return
visit reflects the previous session before the model is involved at all.

## 4. Verification
The repository ships with `typescript.ignoreBuildErrors` and `eslint.ignoreDuringBuilds` enabled in `next.config.ts`, so a passing build alone proves very little. Each check was therefore run **against a baseline** rather than in isolation.

| Check | Baseline (branch point) | After my changes | Verdict |
|---|---|---|---|
| `npx tsc --noEmit` | 74 errors | **73 errors** | **0 new, 1 removed** |
| `npx eslint` (changed files) | 4 warnings, 0 errors | 4 warnings, 0 errors | **0 new** |
| `npm run build` | passes | passes, 12/12 pages | `/api/session-memory` registered |

### Method

The baseline was obtained by stashing the working tree (`git stash push -u`), re-running `tsc`, and restoring.

`SimulatedSessionInterface.tsx` carried two pre-existing errors at the branch point. One of them is now gone:

| | Baseline | After | Error |
|---|---|---|---|
| `ageRange` assignment | line 804 | **removed** | TS2345 |
| `openings[concern]` index | line 1226 | line 1537 | TS7053 |

The `ageRange` error was a bad cast — `option.value as string`, assigned to a field typed `[number, number]` — inside the old age selector. Replacing that selector with the age-group one removed it, which is why the total drops from 74 to 73. The remaining error is untouched by this work.

### A bug found during verification
`resetSession()` cleared the session state but not the new continuity state. Pressing **New Session** after finishing one would therefore have carried the previous patient's memories into the next patient, who would have greeted the counsellor about a session that never happened. Fixed before the branch was pushed.

## 5. Environment Issues and Code Changes

| Issue | Resolution |
|---|---|
| **Node.js was not installed** on the server; `npx` and `npm` were unavailable, so nothing could be type-checked or built | Installed Node v20.18.1 under `~/.local/node` and put it on `PATH`. No root access needed |
| **74 pre-existing TypeScript errors.** `next.config.ts` sets `ignoreBuildErrors: true`, so they do not block a build and a naive run of `tsc` looks alarming | Established the count as a baseline before changing anything, and compared after (§4) |
| **`npm run build` failed at page-data collection** with *"Missing credentials… set the `OPENAI_API_KEY`"* — the API routes instantiate the OpenAI client at module scope | Pre-existing and unrelated to this work. Rebuilt with placeholder environment values to verify the build end to end |
| **Build then failed prerendering `/admin-ultra-secret-admin-2024`** with `auth/invalid-api-key`, because the Firebase config was absent | Also pre-existing. Supplied placeholder Firebase values; the build then completed 12/12 pages |
| **`firestore.rules` had no rule for `trainingSessions`** at all, although the app writes to it. Firestore denies by default when no rule matches | Added rules for both `trainingSessions` and the new `simulatedPatients`, scoped so a counsellor can only read and write their own records |
| **No `.env` template existed** in the repository, so required variables had to be read out of `src/config/env.ts` | Added `.env.local.example` listing every variable with empty values |

### Two pre-existing issues found while standing the app up

Neither is caused by this work, and neither is fixed by it. Both surfaced because deploying to a clean
Firebase project exercises paths that an already-configured project hides.

**1. The security rules gate on a custom claim that nothing sets.** `firestore.rules` tests
`request.auth.token.userType` in eleven places, but no code anywhere sets a custom claim — that
requires the Admin SDK, and `firebase-admin` is not a dependency. `authService.signUp` writes
`userType` into the user's Firestore **document** only. On a clean project every one of those eleven
clauses therefore evaluates false. The practical effect: a counsellor cannot read an unclaimed
conversation, so the real-patient queue appears empty. The rules I added for `simulatedPatients` and
`trainingSessions` test `counselorId == request.auth.uid` instead, so the simulated trainer is
unaffected.

**2. The OpenAI key is shipped to the browser.** Three services call OpenAI directly from the client
with `dangerouslyAllowBrowser: true` — `patientSimulationService.ts:20`,
`conversationAnalysisService.ts:13`, `aiAnalysisService.ts:17` — reading
`NEXT_PUBLIC_OPENAI_API_KEY` (`src/config/env.ts:6`). In Next.js the `NEXT_PUBLIC_` prefix inlines a
value into the client bundle at build time, so the key is served as plain text in a `.js` file. I
confirmed this by building with a placeholder and finding it in `.next/static/chunks/`. The original
author flagged the same thing in a comment (*"In production, move this to backend"*). For this demo I
used a throwaway key; before any deployment these three calls need to move behind an API route.

### Firestore rules added

Session summaries quote the transcript, so they are treated as the counsellor's private practice records rather than shared data:

```
match /simulatedPatients/{patientId} {
  allow create: if request.auth != null &&
    request.auth.uid == request.resource.data.counselorId;
  allow read, update, delete: if request.auth != null &&
    resource.data.counselorId == request.auth.uid;
}
```

## 6. Deviations from the Recommended Steps
The brief includes a suggested procedure (Phases 1–4) while stating that *"you need not follow too closely what these instructions say and can use your own approach."* The deviations are listed here for transparency.

| Recommended | Used | Reason |
|---|---|---|
| Clone `kevalsakhiya/chat-bots` as the starting point | **Not used** | The repository is **149 lines** in total: two standalone chatbot scripts with no login, no database, no pages, no personas and no feedback. The instructor directed this lab at the existing AIMES codebase (~21,500 lines), which already implements five of the seven requirements. Starting from the smaller repository would have meant rebuilding working features |
| Streamlit | **Next.js 15** | Determined by the existing codebase. (Note: `chat-bots` in fact uses **Chainlit**, not Streamlit) |
| Gemini (`gemini-1.5-pro` / `flash`), OpenAI optional | **OpenAI `gpt-4o`** | Every existing API route in the codebase uses `gpt-4o`. Introducing a second provider for one route would have made the feedback and summary paths inconsistent with each other. The brief permits ignoring OpenAI; it does not require Gemini |
| A sidebar model selector | **Not added** | Would only be meaningful with two providers wired up; see above |
| 3–4 preset personas in a dictionary / JSON file | **Already present** | `PatientSimulationService` generates personas from cultural background, gender, age range and concern, with backstory, communication style, emotional expression and trust level derived from them |

## 7. Screenshots — full captures

Captured against a clean Firebase project (`aimes-demo`) created for this report, so every record shown was produced by this run rather than by pre-existing data.

Each feature also appears cropped beside the section that describes it (§2.3 to §2.5). These are the uncropped pages, so the crops can be checked against the context they came from.

### 7.1 Choosing a new simulated patient

![Patient selection](screenshots/01-new-patient-selection.png)

The selection screen in its place on the dashboard. The concern and age-group controls are shown close up in §2.3; what this adds is the surrounding context — the **Simulated Patients** mode card is selected above, and the **Returning patient** tab beside **New patient** already carries a count, which is the entry point for §7.3.

### 7.2 The summary produced when the session ends

![Session summary panel](screenshots/03-session-summary-panel.png)

This is the capture the §2.4 crop comes from. The transcript is above and the record extracted from it is directly below, which is the point: the summary is derived from that conversation rather than being the conversation. For this session with Sam it produced:

| Field | Extracted |
|---|---|
| Headline | *Sam feels overwhelmed by familial and societal expectations.* |
| Ended feeling | *Sam feels nervous but ready to start addressing pressures.* |
| Discussed | anxiety due to family and societal expectations; pressure from being the first in family to attend college |
| New disclosures | first in their family to attend college; feels pressure to be the model minority; experiences anxiety and sleepless nights |
| Agreed next steps | counsellor will support Sam through balancing personal goals and cultural expectations |
| Left unresolved | explore Sam's daily routine and class challenges; discuss how to manage energy and expectations |
| Cultural factors | Sam is Asian-American and feels societal pressure to excel; family sacrifices and cultural expectations are significant stressors |
| Rapport | high |

Two things in this capture are worth pointing at.

**Risk is stated, not omitted.** The panel reads *"No risk indicators recorded in this session."* An empty risk list is information — it says the check ran rather than that it was skipped.

**The culturally specific material survives extraction.** In the transcript Sam reaches for a jade-carving metaphor (*"a jade stone needs to go through a long process of cutting, chiselling, and polishing to become a gem"*) and the Chinese idiom *chī kǔ*, glossed in-line as *"eating bitterness"*. Those are exactly the details a generic summariser flattens into "discussed family pressure". They come through under **Cultural factors** instead, which is the field that then shapes how the patient opens next time.

### 7.3 The patient is now on the roster

![Returning patient list](screenshots/04-returning-patient-list.png)

Before this change no such screen existed: every session began with a stranger. Two patients are listed here — Sam from §7.2, collapsed, and Jordan with the history expanded (shown close up in §2.5). Each card carries the session count, when the patient was last seen, the rapport badge, the previous headline, what was left unresolved, and an expandable history of every prior session.

### 7.4 The second session opens differently

![Second session opening](screenshots/05-second-session-opening.png)

Jordan's card in §7.3 records **High rapport**, and this is the session that follows it. The opening turn is selected by the application from the stored rapport level before the model is called at all, so the first turn of a return visit reflects the previous session deterministically rather than by chance. The line itself is quoted in §2.5.

### Not captured

The real-patient resume path discussed in §1.1 is evidenced by code reference rather than by a screenshot. On a clean Firebase project the counsellor's queue is empty because of the custom-claim gap described in §5, so there is no claimed conversation to reopen. On the lab's configured project that path works as described.

## 8. Deliverables

### Code
| Item | Location |
|---|---|
| Branch for this lab | [`feat/returning-patients-session-memory`](https://github.com/shinnew9/aimes/tree/feat/returning-patients-session-memory) |
| Reviewable diff | [compare view](https://github.com/shinnew9/aimes/compare/fix/spanish-english-translation-toggle...feat/returning-patients-session-memory) |
| Base codebase (Efua Yankey) | [efuayankey/aimes](https://github.com/efuayankey/aimes) |


### Files changed
| File | | Lines | Purpose |
|---|---|---|---|
| `src/components/counselor/SimulatedSessionInterface.tsx` | modified | +486 | New / returning tabs, continuity wiring, end-of-session extraction, multi-concern and age-group selectors, summary panel |
| `src/components/counselor/ReturningPatientList.tsx` | new | +230 | The "existing patients" screen |
| `src/services/sessionMemoryService.ts` | new | +175 | Extraction, normalisation of the model's output, continuity prompt block |
| `src/services/patientRosterService.ts` | new | +156 | Firestore persistence for saved patients |
| `src/app/api/session-memory/route.ts` | new | +128 | Extraction prompt and JSON schema for the carry-forward record |
| `src/components/counselor/SessionSummaryPanel.tsx` | new | +123 | The end-of-session summary panel |
| `src/types/SimulatedPatient.ts` | modified | +97 | `SessionMemory`, `StoredSimulatedPatient`, `AGE_GROUPS`, `concernsOf`, `lifeStageOf` |
| `src/services/patientSimulationService.ts` | modified | +59 | Multi-concern generation, age-group resolution, life stage in the persona prompt |
| `firestore.rules` | modified | +18 | Rules for `simulatedPatients` and `trainingSessions` |
| `.env.local.example` | new | +18 | Template for the required environment variables |

**10 files · 1,403 insertions · 87 deletions**


---

### What is still missing

| | |
|---|---|
| The within-session context cap (`.slice(-4)`) | Untouched. The continuity record is appended to the system prompt and is not subject to it, but the turn-by-turn history still is. §1.2 |
| The custom-claim gap in `firestore.rules` | Documented, not fixed. It blocks the real-patient queue on a clean Firebase project. §5 |
| The OpenAI key in the client bundle | Documented, not fixed. Three services need to move behind an API route. §5 |

### A note on what this app is
Every "patient" in it is an AI-generated persona, not a real person, and nothing the app produces is clinical advice. It exists so that counsellors in training can practise without practising on someone.